In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
!pip install geopandas osmnx pandas shapely pyogrio folium rasterio rioxarray xarray matplotlib networkx requests python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 44.4 MB/s eta 0:00:00
  Attempting uninstall: xarray
    Found existing installation: xarray 2025.12.0
    Uninstalling xarray-2025.12.0:
      Successfully uninstalled xarray-2025.12.0


In [11]:
%cd /content/Nvidia_AI_Models
!git pull origin Phase1_AI

/content/Nvidia_AI_Models
From https://github.com/KVishal1012/Nvidia_AI_Models
 * branch            Phase1_AI  -> FETCH_HEAD
Already up to date.


In [7]:
!find /content/drive/MyDrive/chennai-flood-mvp/data/raw -maxdepth 1 -type f

In [13]:
%%writefile /content/Nvidia_AI_Models/phase1_lab/src/config.py
from pathlib import Path

# Project root inside cloned GitHub repo
PROJECT_ROOT = Path(__file__).resolve().parents[1]

# Google Drive data root for Colab
DRIVE_ROOT = Path("/content/drive/MyDrive/chennai-flood-mvp")

# Data folders
DATA_DIR = DRIVE_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = DATA_DIR / "outputs"

# Area of interest
CITY_NAME = "Chennai"
COUNTRY = "India"
CRS_WGS84 = "EPSG:4326"
CRS_PROJECTED = "EPSG:32644"

# Input file paths
AOI_FILE = RAW_DIR / "chennai_aoi.geojson"
ROADS_FILE = RAW_DIR / "chennai_roads.geojson"
BUILDINGS_FILE = RAW_DIR / "chennai_buildings.geojson"
DRAINAGE_FILE = RAW_DIR / "chennai_drainage.geojson"
RAINFALL_FILE = RAW_DIR / "chennai_rainfall.csv"

# Processed file paths
FLOOD_MASK_FILE = PROCESSED_DIR / "flood_mask.tif"
FLOOD_EXTENT_FILE = PROCESSED_DIR / "flood_extent.geojson"
FLOODED_ROADS_FILE = PROCESSED_DIR / "flooded_roads.geojson"
EXPOSED_BUILDINGS_FILE = PROCESSED_DIR / "exposed_buildings.geojson"
DRAINAGE_RISK_FILE = PROCESSED_DIR / "drainage_risk.geojson"

# Output file paths
FINAL_JSON_FILE = OUTPUT_DIR / "flood_risk_summary.json"
FINAL_GEOJSON_FILE = OUTPUT_DIR / "flood_risk_features.geojson"

# MVP thresholds
SAR_WATER_THRESHOLD_DB = -17
RAINFALL_HIGH_RISK_MM_24H = 150
RAINFALL_MEDIUM_RISK_MM_24H = 80
BUILDING_EXPOSURE_BUFFER_M = 10
ROAD_FLOOD_BUFFER_M = 5

Overwriting /content/Nvidia_AI_Models/phase1_lab/src/config.py


In [14]:
!grep CITY_NAME /content/Nvidia_AI_Models/phase1_lab/src/config.py

CITY_NAME = "Chennai"


In [3]:
!find /content/drive/MyDrive/chennai-flood-mvp/data/raw -maxdepth 1 -type f

/content/drive/MyDrive/chennai-flood-mvp/data/raw/chennai_aoi.geojson
/content/drive/MyDrive/chennai-flood-mvp/data/raw/chennai_roads.geojson
/content/drive/MyDrive/chennai-flood-mvp/data/raw/chennai_buildings.geojson
/content/drive/MyDrive/chennai-flood-mvp/data/raw/chennai_drainage.geojson
/content/drive/MyDrive/chennai-flood-mvp/data/raw/chennai_rainfall.csv


In [3]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

raw_dir = Path("/content/drive/MyDrive/chennai-flood-mvp/data/raw")

aoi = gpd.read_file(raw_dir / "chennai_aoi.geojson")
roads = gpd.read_file(raw_dir / "chennai_roads.geojson")
buildings = gpd.read_file(raw_dir / "chennai_buildings.geojson")
drainage = gpd.read_file(raw_dir / "chennai_drainage.geojson")
rainfall = pd.read_csv(raw_dir / "chennai_rainfall.csv")

print("AOI:", len(aoi))
print("Road segments:", len(roads))
print("Buildings:", len(buildings))
print("Drainage features:", len(drainage))
print("Rainfall rows:", len(rainfall))

AOI: 1
Road segments: 174057
Buildings: 277013
Drainage features: 291
Rainfall rows: 5


In [4]:
import folium
import geopandas as gpd
import pandas as pd

# Keep only geometry and simple string columns for map preview
roads_map = roads.head(1000).copy()
drainage_map = drainage.copy()

# Convert non-geometry columns to string to avoid JSON serialization errors
for col in roads_map.columns:
    if col != "geometry":
        roads_map[col] = roads_map[col].astype(str)

for col in drainage_map.columns:
    if col != "geometry":
        drainage_map[col] = drainage_map[col].astype(str)

# Make sure all layers are EPSG:4326
aoi_map = aoi.to_crs("EPSG:4326")
roads_map = roads_map.to_crs("EPSG:4326")
drainage_map = drainage_map.to_crs("EPSG:4326")

center = aoi_map.geometry.iloc[0].centroid

m = folium.Map(
    location=[center.y, center.x],
    zoom_start=11
)

folium.GeoJson(
    aoi_map,
    name="Chennai AOI",
    style_function=lambda x: {
        "fillColor": "none",
        "color": "blue",
        "weight": 2
    }
).add_to(m)

folium.GeoJson(
    roads_map,
    name="Roads sample",
    style_function=lambda x: {
        "color": "gray",
        "weight": 1
    }
).add_to(m)

folium.GeoJson(
    drainage_map,
    name="Drainage / waterways",
    style_function=lambda x: {
        "color": "cyan",
        "weight": 2
    }
).add_to(m)

folium.LayerControl().add_to(m)

m